# 02. Pollution & DSC Scoring (Image Cell)

**Phase 1**: 5종 polluter × 6 level × 3 데이터셋 → DSC 점수 측정.

split-first 원칙: train/test 분할 → train에만 polluter 적용. (torchvision의 train/test split 그대로 사용 — 별도 split 불필요)

---

In [1]:
# ============================================================
# 0-1. 환경 + 데이터 로드
# ============================================================
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torchvision

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image'
POLLUTED_DIR = f'{BASE}/data/image_polluted'
os.makedirs(POLLUTED_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

%pip install -q timm imagehash opencv-python-headless

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# 0-2. 사전등록 (DATASETS, POLLUTION_LEVELS)
# ============================================================
# ADR-014 사전등록 원본: dataset 3종 (CIFAR10/FashionMNIST/Flowers102), level 6단계.
# Phase 1 정식 run: dataset 2종 (튜닝=CIFAR10 / held-out=FashionMNIST), level 5단계.
# 사전등록 원본 setting은 DATASETS_FULL/POLLUTION_LEVELS_FULL로 보존.

DATASETS_FULL = {
    'CIFAR10': {'loader': 'CIFAR10', 'n_classes': 10, 'image_size': 32, 'channels': 3},
    'FashionMNIST': {'loader': 'FashionMNIST', 'n_classes': 10, 'image_size': 28, 'channels': 1},
    'Flowers102': {'loader': 'Flowers102', 'n_classes': 102, 'image_size': 224, 'channels': 3},
}
POLLUTION_LEVELS_FULL = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95]

DATASETS = {k: v for k, v in DATASETS_FULL.items() if k in ('CIFAR10', 'FashionMNIST')}
POLLUTION_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
print(f'Phase 1: datasets={list(DATASETS.keys())}, levels={POLLUTION_LEVELS}')

RANDOM_SEED = 42
SAMPLE_CAP = 5000  # 폴루션 후 DSC 계산 시 sample_cap

def load_train(ds_name):
    if ds_name == 'CIFAR10':
        return torchvision.datasets.CIFAR10(f'{DATA_DIR}/CIFAR10', train=True, download=True)
    if ds_name == 'FashionMNIST':
        return torchvision.datasets.FashionMNIST(f'{DATA_DIR}/FashionMNIST', train=True, download=True)
    if ds_name == 'Flowers102':
        return torchvision.datasets.Flowers102(f'{DATA_DIR}/Flowers102', split='train', download=True)

print(f'데이터셋: {list(DATASETS.keys())}, 강도: {POLLUTION_LEVELS}')


Phase 1: datasets=['CIFAR10', 'FashionMNIST'], levels=[0.1, 0.3, 0.5, 0.7, 0.9]
데이터셋: ['CIFAR10', 'FashionMNIST'], 강도: [0.1, 0.3, 0.5, 0.7, 0.9]


## 1. Polluter 5종 + DSC 점수

In [3]:
# ============================================================
# 1-1. polluter import + DSC import
# ============================================================
import os, sys, importlib
BASE = '/content/drive/MyDrive/capstone/dsc'

# Drive 마운트 stale 자동 복구 (어제 세션 끊긴 후 잔재 대비)
if not os.path.isdir(f'{BASE}/dsc_framework'):
    print('dsc_framework 디렉토리 안 보임 — drive force_remount 시도...')
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

assert os.path.isdir(f'{BASE}/dsc_framework'), (
    f'force_remount 후에도 {BASE}/dsc_framework 없음. Drive 동기화 확인 필요.'
)

# sys.path 보강 (cell 0-1 미실행 대비)
if BASE not in sys.path:
    sys.path.insert(0, BASE)

# import 캐시 꼬임 클리어
importlib.invalidate_caches()
for _mod in list(sys.modules):
    if _mod.startswith('dsc_framework'):
        del sys.modules[_mod]

from dsc_framework import compute_dsc_image
from dsc_framework.image_polluters import (
    CompletenessImagePolluter, NoiseInjectionPolluter, BlurPolluter,
    ClassBalanceImagePolluter, LabelSwapPolluter,
)

def create_polluters(level, seed=RANDOM_SEED):
    return [
        ('completeness_image', CompletenessImagePolluter(level=level, random_seed=seed)),
        ('noise_injection', NoiseInjectionPolluter(level=level, random_seed=seed)),
        ('blur', BlurPolluter(level=level, random_seed=seed)),
        ('class_balance', ClassBalanceImagePolluter(level=level, random_seed=seed)),
        ('label_swap', LabelSwapPolluter(level=level, random_seed=seed)),
    ]
print('Polluter 5종 정의 완료')

Polluter 5종 정의 완료


In [4]:
# ============================================================
# 1-2. dataset → numpy 변환 + 폴루션 적용 + DSC
# ============================================================
def dataset_to_arrays(ds, sample_cap=None, random_state=1):
    images, labels = [], []
    n = len(ds) if sample_cap is None else min(len(ds), sample_cap)
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(ds))[:n] if sample_cap else range(n)
    for i in idx:
        img, lbl = ds[i]
        images.append(np.array(img))
        labels.append(int(lbl))
    return images, labels


from time import time

dsc_rows = []
total_start = time()

for ds_name in DATASETS:
    print(f'\n=== {ds_name} ===')
    train_ds = load_train(ds_name)
    images_clean, labels_clean = dataset_to_arrays(train_ds, sample_cap=SAMPLE_CAP, random_state=1)
    print(f'  loaded {len(images_clean)} images')

    # baseline DSC
    res_base = compute_dsc_image(images_clean, labels_clean, sample_cap=SAMPLE_CAP)
    print(f'  baseline DSC = {res_base["score"]} ({res_base["grade"]})')
    dsc_rows.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0, **res_base})

    # 폴루션
    for level in POLLUTION_LEVELS:
        for polluter_name, polluter in create_polluters(level):
            t0 = time()
            try:
                pi, pl = polluter.pollute(images_clean, labels_clean)
                res_p = compute_dsc_image(pi, pl, sample_cap=SAMPLE_CAP)
                # 폴루션 데이터를 디스크에 저장 (03 노트북이 다시 사용)
                pol_dir = f'{POLLUTED_DIR}/{ds_name}/{polluter_name}_{int(level*100)}'
                os.makedirs(pol_dir, exist_ok=True)
                arrs = [np.asarray(img, dtype=np.uint8) for img in pi]
                shapes = {a.shape for a in arrs}
                images_arr = np.stack(arrs) if len(shapes) == 1 else np.array(arrs, dtype=object)
                np.savez_compressed(f'{pol_dir}/data.npz', images=images_arr, labels=np.array(pl))
                dsc_rows.append({'dataset': ds_name, 'polluter': polluter_name, 'level': level, **res_p})
                elapsed = time() - t0
                print(f'  {polluter_name:<22s} L={level:.2f}  DSC={res_p["score"]:6.2f}  Δ={res_p["score"]-res_base["score"]:+.2f}  ({elapsed:.0f}s)')
            except Exception as e:
                print(f'  {polluter_name:<22s} L={level:.2f}  ERROR: {e}')

print(f'\n총 {len(dsc_rows)}건 ({time() - total_start:.0f}초)')


=== CIFAR10 ===
  loaded 5000 images
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 169MB/s]


  baseline DSC = 91.35 (A)
  completeness_image     L=0.10  DSC= 90.03  Δ=-1.32  (32s)
  noise_injection        L=0.10  DSC= 90.03  Δ=-1.32  (34s)
  blur                   L=0.10  DSC= 89.79  Δ=-1.56  (32s)
  class_balance          L=0.10  DSC= 91.17  Δ=-0.18  (30s)
  label_swap             L=0.10  DSC= 88.58  Δ=-2.77  (32s)
  completeness_image     L=0.30  DSC= 87.84  Δ=-3.51  (31s)
  noise_injection        L=0.30  DSC= 86.16  Δ=-5.19  (33s)
  blur                   L=0.30  DSC= 77.75  Δ=-13.60  (32s)
  class_balance          L=0.30  DSC= 90.79  Δ=-0.56  (22s)
  label_swap             L=0.30  DSC= 83.86  Δ=-7.49  (31s)
  completeness_image     L=0.50  DSC= 85.98  Δ=-5.37  (30s)
  noise_injection        L=0.50  DSC= 84.07  Δ=-7.28  (32s)
  blur                   L=0.50  DSC= 74.44  Δ=-16.91  (33s)
  class_balance          L=0.50  DSC= 89.90  Δ=-1.45  (20s)
  label_swap             L=0.50  DSC= 80.84  Δ=-10.51  (30s)
  completeness_image     L=0.70  DSC= 84.15  Δ=-7.20  (30s)
  noise_in

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  blur                   L=0.90  DSC= 83.22  Δ=-3.47  (30s)
  class_balance          L=0.90  DSC= 80.26  Δ=-6.43  (9s)
  label_swap             L=0.90  DSC= 69.69  Δ=-17.00  (29s)

총 52건 (1476초)


In [5]:
# ============================================================
# 2-3. 결과 저장
# ============================================================
df_dsc = pd.DataFrame(dsc_rows)
out_path = f'{RESULTS_DIR}/dsc_scores_image.csv'
df_dsc.to_csv(out_path, index=False)
print(f'DSC 점수 저장: {out_path} (총 {len(df_dsc)}건)')
print('--- 노트북 02 이미지 cell 완료 ---')
df_dsc.head(15)

DSC 점수 저장: /content/drive/MyDrive/capstone/dsc/results/dsc_scores_image.csv (총 52건)
--- 노트북 02 이미지 cell 완료 ---


,dataset,polluter,level,score,grade,completeness_image,uniqueness,validity,consistency,outlier_ratio,class_balance,sample_quality_image,feature_correlation,label_consistency,feature_informativeness
0,CIFAR10,none,0.0,91.35,A,0.9966,1.0,1.0,1.0,0.9870,0.9480,0.9357,1.0,0.6476,1.0
1,CIFAR10,completeness_image,0.1,90.03,A,0.9684,1.0,1.0,1.0,0.9880,0.9480,0.9408,1.0,0.5985,1.0
2,CIFAR10,noise_injection,0.1,90.03,A,0.9985,1.0,1.0,1.0,0.9874,0.9480,0.9362,1.0,0.5795,1.0
3,CIFAR10,blur,0.1,89.79,B,0.9974,1.0,1.0,1.0,0.9870,0.9480,0.9233,1.0,0.5784,1.0
4,CIFAR10,class_balance,0.1,91.17,A,0.9967,1.0,1.0,1.0,0.9862,0.9365,0.9348,1.0,0.6449,1.0
5,CIFAR10,label_swap,0.1,88.58,B,0.9966,1.0,1.0,1.0,0.9870,0.9440,0.9357,1.0,0.5109,1.0
6,CIFAR10,completeness_image,0.3,87.84,B,0.9121,1.0,1.0,1.0,0.9908,0.9480,0.9515,1.0,0.5226,1.0
7,CIFAR10,noise_injection,0.3,86.16,B,0.9981,1.0,1.0,1.0,0.9890,0.9480,0.9390,1.0,0.3839,1.0
8,CIFAR10,blur,0.3,77.75,B,0.9986,1.0,1.0,1.0,0.9870,0.9480,0.5239,1.0,0.2747,1.0
9,CIFAR10,class_balance,0.3,90.79,A,0.9967,1.0,1.0,1.0,0.9866,0.9056,0.9331,1.0,0.6429,1.0
